# LensIQ: deploy a single-purpose Roboflow detector to Model Serving

Logs a tiny MLflow PyFunc that proxies one Roboflow Universe model,
registers it to Unity Catalog, and serves it behind its own Databricks
Model Serving endpoint. Re-run this notebook once per use case (license
plate, spill, wet floor sign, cigarette/vape, slip & fall) with different
widgets - the bundle's `lensiq_deploy_roboflow_detectors` job wires that
up as a multi-task fan-out.

Why one endpoint per use case (instead of one PyFunc dispatching by
`model_id`):

- Independent UC registry entry per model -> per-use-case versioning + RBAC.
- Independent serving config (workload size, scale-to-zero, alerts).
- Single-purpose endpoints make ownership / cost-attribution obvious.
- One endpoint can be updated without redeploying the others.

Why a PyFunc instead of hitting Roboflow directly from the app:

Roboflow Universe public models can't be downloaded for offline serving,
so we proxy. The PyFunc gives us a managed Databricks endpoint (auth,
scale-to-zero, logging, billing) while keeping the app code unaware of
Roboflow as a vendor.

Payload (matches AppKit `serving()` invoke):

```json
{"dataframe_records": [{"image": "<b64>", "conf": 0.35}]}
```

Response (matches the YOLO endpoint so `_normalizeDatabricks` works):

```json
{"predictions": [[{"label": "...", "class_id": 0, "confidence": 0.8, "bbox": [x1,y1,x2,y2]}]]}
```

In [ ]:
dbutils.widgets.text("catalog", "iot_dev")
dbutils.widgets.text("schema", "lensiq")
# Per-use-case slug. Used to build both the UC registered model name
# (`lensiq_<slug>`) and the serving endpoint name (`lensiq-<slug-with-dashes>`).
# Match the model ids declared in client/src/lib/models.ts so the AppKit
# server can bind the same alias.
dbutils.widgets.text("model_slug", "license_plate")
dbutils.widgets.text("model_display_name", "License plates")
# Roboflow Universe slug (project/version, no workspace prefix - that's
# what serverless.roboflow.com routes by).
dbutils.widgets.text("roboflow_model_id", "license-plate-recognition-rxg4e/13")
dbutils.widgets.text("api_key_scope", "reggie_pierce")
dbutils.widgets.text("api_key_secret", "ROBOFLOW_API_KEY")

In [ ]:
%pip install -q mlflow>=2.13 requests
dbutils.library.restartPython()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_roboflow_detector")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
MODEL_SLUG = dbutils.widgets.get("model_slug").strip()
MODEL_DISPLAY_NAME = dbutils.widgets.get("model_display_name").strip() or MODEL_SLUG
ROBOFLOW_MODEL_ID = dbutils.widgets.get("roboflow_model_id").strip()
API_KEY_SCOPE = dbutils.widgets.get("api_key_scope")
API_KEY_SECRET = dbutils.widgets.get("api_key_secret")

# Conventions tying everything together:
#   - UC registered model:  <catalog>.<schema>.lensiq_<slug>
#   - Serving endpoint:     lensiq-<slug-with-dashes>
# The AppKit server expects matching aliases (see server/server.ts).
REGISTERED = f"{CATALOG}.{SCHEMA}.lensiq_{MODEL_SLUG}"
ENDPOINT = f"lensiq-{MODEL_SLUG.replace('_', '-')}"

# Sanity-check the Roboflow key exists before we waste time logging a model.
API_KEY = dbutils.secrets.get(scope=API_KEY_SCOPE, key=API_KEY_SECRET)
if not API_KEY:
    raise RuntimeError(
        f"Roboflow API key not found at secrets/{API_KEY_SCOPE}/{API_KEY_SECRET}"
    )

LOG.info("Deploying Roboflow proxy: %s -> %s (model_id=%s)",
         MODEL_DISPLAY_NAME, ENDPOINT, ROBOFLOW_MODEL_ID)
LOG.info("  registered_model=%s", REGISTERED)

## PyFunc wrapper

Single-model proxy. The Roboflow `project/version` slug is baked into the
MLflow artifact via `model_config` so the served endpoint can't be tricked
into proxying a different model. The Roboflow API key is read from the
endpoint environment (wired below via `environment_vars`).

In [ ]:
import json

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models import infer_signature


class RoboflowDetector(mlflow.pyfunc.PythonModel):
    """Single-model PyFunc that proxies inference to one Roboflow Universe
    model. The model id is baked into the MLflow artifact via
    `model_config["roboflow_model_id"]` so the deployed endpoint always
    targets exactly that model.

    Inputs (per row):
      - image: base64-encoded JPEG/PNG (with or without `data:` prefix).
      - conf:  optional confidence threshold, default 0.35.

    Output (per row): list of `{label, class_id, confidence, bbox}` with
    `bbox` as `[x1, y1, x2, y2]`. Matches the YOLO endpoint's contract.
    """

    ROBOFLOW_BASE = "https://serverless.roboflow.com"

    def load_context(self, context):
        import os as _os
        self._api_key = _os.environ.get("ROBOFLOW_API_KEY", "")
        self._model_id = context.model_config["roboflow_model_id"]

    def _strip_data_url(self, image_b64):
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            return image_b64.split(",", 1)[1]
        return image_b64 or ""

    def _run_one(self, image_b64, conf):
        if not image_b64:
            return []
        if not self._api_key:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": "ROBOFLOW_API_KEY env var is not set on endpoint"}]

        import requests
        # Roboflow's REST API expects confidence as a fraction in [0, 1]
        # (matches the OpenAPI default of 0.4). Do NOT scale to percentage -
        # that gets interpreted as a threshold of 35.0 which filters out
        # every prediction.
        c_frac = max(0.01, min(0.99, float(conf if conf is not None else 0.35)))
        url = f"{self.ROBOFLOW_BASE}/{self._model_id}"
        try:
            resp = requests.post(
                url,
                params={
                    "api_key": self._api_key,
                    "confidence": c_frac,
                    "format": "json",
                },
                headers={"Content-Type": "application/x-www-form-urlencoded"},
                data=self._strip_data_url(image_b64),
                timeout=30,
            )
            resp.raise_for_status()
        except Exception as ex:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": f"roboflow http error: {ex}"}]

        body = resp.json()
        out = []
        for p in body.get("predictions", []) or []:
            try:
                cx, cy = float(p["x"]), float(p["y"])
                w, h = float(p["width"]), float(p["height"])
            except (KeyError, TypeError, ValueError):
                continue
            out.append({
                "label": p.get("class") or p.get("class_name") or "object",
                "class_id": int(p.get("class_id", -1)),
                "confidence": float(p.get("confidence", 0.0)),
                "bbox": [
                    int(round(cx - w / 2)),
                    int(round(cy - h / 2)),
                    int(round(cx + w / 2)),
                    int(round(cy + h / 2)),
                ],
            })
        return out

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(r.get("image"), r.get("conf")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([
    {"image": _TINY_PNG_B64, "conf": 0.35},
])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

with mlflow.start_run(run_name=f"deploy_roboflow_{MODEL_SLUG}") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RoboflowDetector(),
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        model_config={"roboflow_model_id": ROBOFLOW_MODEL_ID},
        pip_requirements=[
            "mlflow>=2.13",
            "requests>=2.31",
        ],
    )
    mlflow.set_tag("lensiq.detector_slug", MODEL_SLUG)
    mlflow.set_tag("lensiq.display_name", MODEL_DISPLAY_NAME)
    mlflow.set_tag("lensiq.roboflow_model_id", ROBOFLOW_MODEL_ID)
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

Pulls `ROBOFLOW_API_KEY` from the configured Databricks secret using
`environment_vars` so the served PyFunc reads it from `os.environ` at
load time without baking the value into the artifact.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
    environment_vars={
        "ROBOFLOW_API_KEY": f"{{{{secrets/{API_KEY_SCOPE}/{API_KEY_SECRET}}}}}",
    },
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)